In [ ]:
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

DATA_PATH = Path(r'C:\Users\eszti\Downloads\archive\data.csv')
TARGET_COL = 'value'
RANDOM_STATE = 7
TEST_SIZE = 0.25

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['row_id', 'day_name', 'day_name_long', 'data_uploaded'])
df['facility_code'] = df['facility_code'].astype(str)
df['time_band'] = df['time_band'].astype(float)
df['tb_sin'] = np.sin(2 * np.pi * df['time_band'] / 24)
df['tb_cos'] = np.cos(2 * np.pi * df['time_band'] / 24)

cat_features = ['facility_code', 'day_of_week']
num_features = ['time_band', 'tb_sin', 'tb_cos']
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features),
])

models = {
    'LinearRegression': LinearRegression(),
    'SVR_rbf': SVR(kernel='rbf'),
    'DecisionTreeRegressor(max_depth=4)': DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestRegressor(n_estimators=200, max_depth=6)': RandomForestRegressor(n_estimators=200, max_depth=6, random_state=RANDOM_STATE),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=RANDOM_STATE),
    'MLPRegressor(hidden_layer_sizes=(16,16), max_iter=500)': MLPRegressor(hidden_layer_sizes=(16, 16), max_iter=500, random_state=RANDOM_STATE),
}

X = df[['facility_code', 'day_of_week', 'time_band', 'tb_sin', 'tb_cos']]
y = df[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

rows = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred, squared=False),
        'R2': r2_score(y_test, pred),
    })

results = pd.DataFrame(rows).sort_values('R2', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(results['model'], results['R2'], color='steelblue')
plt.xlabel('R²')
plt.ylabel('Model')
plt.title('Model comparison by R²')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(Path.cwd() / 'model_comparison_r2.png', dpi=200)
plt.show()

In [ ]:
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

DATA_PATH = Path(r'C:\Users\eszti\Downloads\archive\data.csv')
TARGET_COL = 'value'
RANDOM_STATE = 7
TEST_SIZE = 0.25

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['row_id', 'day_name', 'day_name_long', 'data_uploaded'])
df['facility_code'] = df['facility_code'].astype(str)
df['time_band'] = df['time_band'].astype(float)
df['tb_sin'] = np.sin(2 * np.pi * df['time_band'] / 24)
df['tb_cos'] = np.cos(2 * np.pi * df['time_band'] / 24)

cat_features = ['facility_code', 'day_of_week']
num_features = ['time_band', 'tb_sin', 'tb_cos']
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features),
])

models = {
    'LinearRegression': LinearRegression(),
    'SVR_rbf': SVR(kernel='rbf'),
    'DecisionTreeRegressor(max_depth=4)': DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestRegressor(n_estimators=200, max_depth=6)': RandomForestRegressor(n_estimators=200, max_depth=6, random_state=RANDOM_STATE),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=RANDOM_STATE),
    'MLPRegressor(hidden_layer_sizes=(16,16), max_iter=500)': MLPRegressor(hidden_layer_sizes=(16, 16), max_iter=500, random_state=RANDOM_STATE),
}

X = df[['facility_code', 'day_of_week', 'time_band', 'tb_sin', 'tb_cos']]
y = df[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

rows = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred, squared=False),
        'R2': r2_score(y_test, pred),
    })

results = pd.DataFrame(rows).sort_values('R2', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(results['model'], results['R2'], color='steelblue')
plt.xlabel('R²')
plt.ylabel('Model')
plt.title('Model comparison by R²')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(Path.cwd() / 'model_comparison_r2.png', dpi=200)
plt.show()
